# Input and load data

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import blosum as bl

from scipy.stats import spearmanr, pearsonr
from collections import defaultdict
from tqdm import tqdm
from omegaconf import OmegaConf
from matplotlib_venn import venn3, venn2

In [ ]:
sns.set_context('talk')
sns.set_palette('colorblind')
palette = sns.color_palette('colorblind')

In [ ]:
config_filepath = 'my/path/to/config.yaml'
cfg = OmegaConf.load(config_filepath)

In [ ]:
cell_lines = cfg['cell_lines']
editors = cfg['editors']
target_genes = cfg['target_genes']
processed_files_dirpath = cfg['processed_files_dirpath']
figures_dirpath = cfg['figures_dirpath']

In [ ]:
with_mut_dfs = []
for gene in target_genes:
    gene_dirpath = os.path.join(processed_files_dirpath, gene)
    df_filepath = os.path.join(gene_dirpath, f'{gene}_all_results.csv')
    df = pd.read_csv(df_filepath)
    with_mut_dfs.append(df)
with_mut_df = pd.concat(with_mut_dfs, ignore_index=True)

In [ ]:
no_dup_guide_dfs = []
for gene in target_genes:
    gene_dirpath = os.path.join(processed_files_dirpath, gene)
    df_filepath = os.path.join(gene_dirpath, f'{gene}_results_no_dup_guide.csv')
    df = pd.read_csv(df_filepath)
    no_dup_guide_dfs.append(df)
no_dup_guide_df = pd.concat(no_dup_guide_dfs, ignore_index=True)

In [ ]:
no_dup_position_dfs = []
for gene in target_genes:
    gene_dirpath = os.path.join(processed_files_dirpath, gene)
    df_filepath = os.path.join(gene_dirpath, f'{gene}_results_no_dup_position.csv')
    df = pd.read_csv(df_filepath)
    no_dup_position_dfs.append(df)
no_dup_position_df = pd.concat(no_dup_position_dfs, ignore_index=True)

In [ ]:
no_dup_location_dfs = []
for gene in target_genes:
    gene_dirpath = os.path.join(processed_files_dirpath, gene)
    df_filepath = os.path.join(gene_dirpath, f'{gene}_results_no_dup_location.csv')
    df = pd.read_csv(df_filepath)
    no_dup_location_dfs.append(df)
no_dup_location_df = pd.concat(no_dup_location_dfs, ignore_index=True)

# Fraction of off-targets gene

In [ ]:
dd = no_dup_guide_df.drop_duplicates('sgRNA_ID')

In [ ]:
n_ot = dd[dd['is_ot'].replace({0: False, 1: True})]['Gene'].value_counts()
n_ot.name = 'OT'
n_guides = dd['Gene'].value_counts()
n_guides.name = 'NGuide'

In [ ]:
n_ot_guides = pd.DataFrame(n_ot).join(pd.DataFrame(n_guides))

In [ ]:
n_ot_guides['FOT'] = n_ot_guides['OT'] / n_ot_guides['NGuide']

In [ ]:
n_ot_guides.sort_values('FOT', ascending=False)

In [ ]:
# Remove OT
with_mut_df = with_mut_df[with_mut_df['is_ot'] != 1].copy()
no_dup_guide_df = no_dup_guide_df[no_dup_guide_df['is_ot'] != 1].copy()
no_dup_position_df = no_dup_position_df[no_dup_position_df['is_ot'] != 1].copy()
no_dup_location_df = no_dup_location_df[no_dup_location_df['is_ot'] != 1].copy()

In [ ]:
# we identified X sgRNAs statistically associated with a reduced cell fitness 
no_dup_guide_df[no_dup_guide_df['lt_cutoff']].drop_duplicates('sgRNA_ID').shape

In [ ]:
no_dup_guide_df[no_dup_guide_df['lt_cutoff']].groupby(['cell_line']).count()['lt_cutoff']

In [ ]:
old_name = 'sgRNA_ID'
new_name = 'sgrna'
for df in [with_mut_df, no_dup_guide_df, no_dup_position_df, no_dup_location_df]:
    df[new_name] = df[old_name]

In [ ]:
# Remove the T47D-CBE combination, which is an outlier regarding controls

no_dup_position_df_filtered = no_dup_position_df[~(
    (no_dup_position_df['cell_line'] == 'T47D')
    & (no_dup_position_df['editor'] == 'CBE')
)]

with_mut_df_filtered = with_mut_df[~(
    (with_mut_df['cell_line'] == 'T47D')
    & (with_mut_df['editor'] == 'CBE')
)]

# Examples of data exploration

In [ ]:
sdf = with_mut_df_filtered[
(with_mut_df_filtered['Gene'] == 'PIK3CA')
&
# (with_mut_df_filtered['Protein_position'] > 2335) & (with_mut_df_filtered['Protein_position'] < 2343)
(with_mut_df_filtered['Protein_position'] == 912)
&
(with_mut_df_filtered['lt_cutoff'])
]
sdf[['sgRNA_ID', 'LFC', 'lt_cutoff', 'editor', 'cell_line', 'Original_AA', 'Protein_position', 'Modified_AA', 'Variant class', 'CRISPR_PAM_Sequence']
].drop_duplicates().sort_values(['Protein_position', 'sgRNA_ID', 'Modified_AA'])

In [ ]:
with_mut_df_filtered[
(with_mut_df_filtered['sgRNA_ID'] == '226PDPK1')
# &
# (with_mut_df_filtered['lt_cutoff'])
][['sgRNA_ID', 'LFC', 'lt_cutoff', 'editor', 'cell_line', 'Original_AA', 'Protein_position', 'Modified_AA', 'Variant class', 'CRISPR_PAM_Sequence','Consequence', 'is_ot']
].drop_duplicates().sort_values(['sgRNA_ID', 'Original_AA', 'Modified_AA', 'Consequence'])

In [ ]:
with_mut_df_filtered[
(with_mut_df_filtered['sgRNA_ID'] == '715MAPKAP1')
# &
# (with_mut_df_filtered['lt_cutoff'])
][['sgRNA_ID', 'LFC', 'lt_cutoff', 'editor', 'cell_line', 'Original_AA', 'Protein_position', 'Modified_AA', 'Variant class', 'CRISPR_PAM_Sequence','Consequence']
].drop_duplicates().sort_values(['sgRNA_ID', 'Original_AA', 'Modified_AA', 'Consequence'])

In [ ]:
with_mut_df_filtered[
(with_mut_df_filtered['Gene'] == 'MAPKAP1')
&
# (with_mut_df_filtered['Protein_position'] > 2335) & (with_mut_df_filtered['Protein_position'] < 2343)
(with_mut_df_filtered['Protein_position'] == 205)
&
(with_mut_df_filtered['lt_cutoff'])
][['sgRNA_ID', 'LFC', 'lt_cutoff', 'editor', 'cell_line', 'Original_AA', 'Protein_position', 'Modified_AA', 'Variant class', 'CRISPR_PAM_Sequence']
].drop_duplicates().sort_values(['Protein_position', 'sgRNA_ID', 'Modified_AA'])

In [ ]:
with_mut_df_filtered[
(with_mut_df_filtered['Gene'] == 'MAPKAP1')
&
# (with_mut_df_filtered['Protein_position'] > 2335) & (with_mut_df_filtered['Protein_position'] < 2343)
# (with_mut_df_filtered['Protein_position'] == 524)
(with_mut_df_filtered['Protein_position'].isin([206, 207, 210, 211, 213, 214, 215]))
&
(with_mut_df_filtered['lt_cutoff'])
][['sgRNA_ID', 'CRISPR_PAM_Sequence', 'Location', 'editor', ]
].drop_duplicates().sort_values(['sgRNA_ID'])

In [ ]:
with_mut_df_filtered[
(with_mut_df_filtered['sgRNA_ID'].isin(['361PDPK1', '642PDPK1', '646PDPK1', '951PDPK1']))
# &
# (with_mut_df_filtered['lt_cutoff'])
][['sgRNA_ID', 'CRISPR_PAM_Sequence', 'Location', 'editor', ]
].drop_duplicates(subset=['sgRNA_ID']).sort_values(['sgRNA_ID']).set_index('sgRNA_ID')

In [ ]:
with_mut_df_filtered[
(with_mut_df_filtered['sgRNA_ID'].isin(['693MAPKAP1', '171MAPKAP1', '713MAPKAP1', '715MAPKAP1']))
# &
# (with_mut_df_filtered['lt_cutoff'])
][['sgRNA_ID', 'CRISPR_PAM_Sequence', 'Location', 'editor', ]
].drop_duplicates(subset=['sgRNA_ID']).sort_values(['sgRNA_ID']).set_index('sgRNA_ID')

In [ ]:
with_mut_df_filtered[
(with_mut_df_filtered['sgRNA_ID'] == '422PIK3CA')
&
(with_mut_df_filtered['lt_cutoff'])
][['sgRNA_ID', 'LFC', 'lt_cutoff', 'editor', 'cell_line', 'Original_AA', 'Protein_position', 'Modified_AA', 'Variant class', 'Consequence', 'Location']].drop_duplicates()

In [ ]:
mtor_df = no_dup_position_df_filtered[no_dup_position_df_filtered['Gene'] == 'MTOR']
gdf = mtor_df.groupby(['editor', 'Protein_position'])['lt_cutoff'].mean()
id_cols = ['Protein_position', 'lt_cutoff']
ascending_id_cols = [True, False]
n_hits = gdf.reset_index().sort_values(id_cols, ascending=ascending_id_cols).drop_duplicates(id_cols)
hits = n_hits[n_hits['lt_cutoff'] == 1.0]['Protein_position'].unique()
', '.join([str(int(i)) for i in hits])

In [ ]:
rheb_df = no_dup_position_df_filtered[no_dup_position_df_filtered['Gene'] == 'RHEB']
gdf = rheb_df.groupby(['editor', 'Protein_position'])['lt_cutoff'].mean()
id_cols = ['Protein_position', 'lt_cutoff']
ascending_id_cols = [True, False]
n_hits = gdf.reset_index().sort_values(id_cols, ascending=ascending_id_cols).drop_duplicates(id_cols)
hits = n_hits[n_hits['lt_cutoff'] == 1.0]['Protein_position'].unique()
', '.join([str(int(i)) for i in hits])

In [ ]:
rheb_df[rheb_df['Protein_position'] == 76]

In [ ]:
gdf

In [ ]:
guides = """2710MTOR
3591MTOR
3564MTOR
3565MTOR
1776MTOR
1778MTOR
3146MTOR
2775MTOR
1242PIK3CA
1308PIK3CA
1304PIK3CA
108PIK3CA
1279PIK3CA
1015PIK3CA
322PIK3CA
871PIK3CA
216PIK3CA
238PIK3CA
261PIK3CA
403PIK3CA
233RHEB
79PIK3CA
422PIK3CA"""
guides = guides.split('\n')

In [ ]:
df = with_mut_df
for guide in guides:
    print(guide)
    print(df[df['sgRNA_ID'] == guide][['editor', 'Original_AA', 'Modified_AA', 'Protein_position', 'Variant class', 'Consequence', 'CRISPR_PAM_Sequence', 'Location']].drop_duplicates())
    print('\n')

In [ ]:
missense_guides = no_dup_guide_df[no_dup_guide_df['main_conseq'] == 'missense']
df = with_mut_df[(with_mut_df['sgrna'].isin(missense_guides['sgrna'].values))
                             & (with_mut_df['Consequence'] == 'missense')]
for guide in guides:
    print(guide)
    print(df[df['sgRNA_ID'] == guide][['editor', 'Original_AA', 'Modified_AA', 'Protein_position', 'Variant class']].drop_duplicates())
    print('')

In [ ]:
gdf

# Start analysis

In [ ]:
no_dup_guide_df['Consequence'].value_counts(dropna=False)

In [ ]:
# high_imp_splice = ['splice_acceptor', 'splice_donor']
# low_imp_splice = ['splice_donor_region', 'splice_region', 'splice_polypyrimidine_tract']
splice = ['splice_acceptor', 'splice_donor', 'splice_donor_region']

consequence_order = ['stop_gained', 'stop_lost', 'start_lost'] \
                    + splice \
                    + ['missense'] \
                    + ['intron'] \
                    + ['synonymous']
                    # + ['stop_retained'] \
                    # + [cons for cons in with_mut_df['Consequence'].unique() 
                    #    if isinstance(cons, str) and 'UTR' in cons and not ',' in cons] \
                    # + ['unknown', 'wt', 'coding_sequence']

In [ ]:
variant_order = ['Total-loss', 'SBI', 'WT-like']
main_conseq_order = ['synonymous', 'missense', 'splice', 'stop_gained', 'start_lost', 
                     # 'unknown'
                    ]
consq_consv_order = ['synonymous', 'missense_conservative', 'low_imp_splice', 'missense_non_conservative', 'high_imp_splice', 'stop']
consq_consv_to_mutation = {'missense_conservative': 'missense conservative',
                          'low_imp_splice': 'low impact splice',
                          'missense_non_conservative': 'missense non conservative',
                          'high_imp_splice': 'high impact splice',
                          'stop': 'stop gained'}
mutation_consequence_order = ['synonymous', 'missense conservative', 'low impact splice', 'missense non conservative', 'high impact splice', 'stop gained']

In [ ]:
no_dup_guide_df['Mutation consequence'] = no_dup_guide_df['consq_consv'].replace(consq_consv_to_mutation)
# no_dup_guide_df['Mutation consequence'] = no_dup_guide_df['Mutation consequence'].apply(lambda s: s.replace('_', ' '))

In [ ]:
with_mut_df['Consequence'] = pd.Categorical(with_mut_df['Consequence'], consequence_order)
no_dup_guide_df['Consequence'] = pd.Categorical(no_dup_guide_df['Consequence'], consequence_order)
no_dup_position_df['Consequence'] = pd.Categorical(no_dup_position_df['Consequence'], consequence_order)
no_dup_location_df['Consequence'] = pd.Categorical(no_dup_location_df['Consequence'], consequence_order)

In [ ]:
with_mut_df['Variant class'] = pd.Categorical(with_mut_df['Variant class'], variant_order)
no_dup_guide_df['Variant class'] = pd.Categorical(no_dup_guide_df['Variant class'], variant_order)
no_dup_position_df['Variant class'] = pd.Categorical(no_dup_position_df['Variant class'], variant_order)
no_dup_location_df['Variant class'] = pd.Categorical(no_dup_location_df['Variant class'], variant_order)

In [ ]:
missense_guides = no_dup_guide_df[no_dup_guide_df['main_conseq'] == 'missense']
missense_df = with_mut_df[(with_mut_df['sgrna'].isin(missense_guides['sgrna'].values))
                             & (with_mut_df['Consequence'] == 'missense')]
missense_df = missense_df.sort_values('LFC').drop_duplicates(['cell_line', 'editor', 'Gene', 'Protein_position'])

In [ ]:
missense_df['AA_Mutation'] = missense_df['Original_AA'] + '/' + missense_df['Modified_AA']

In [ ]:
# Essential genes
with_mut_df[with_mut_df['is_essential']]['Gene'].unique()

In [ ]:
essential_genes_df = no_dup_guide_df[['cell_line', 'Gene', 'is_essential']].drop_duplicates(
    ).sort_values(['cell_line', 'is_essential', 'Gene'], ascending=[True, False, True])
essential_genes_df_filepath = os.path.join(processed_files_dirpath, 'essential_genes_per_cell_line.csv')
essential_genes_df.to_csv(essential_genes_df_filepath, index=False)

# Venn Diagrams

In [ ]:
def get_venn_set_idx(row, editor):
    hgc27 = row['HGC27'] == 1
    mcf7 = row['MCF7'] == 1
    if editor == 'ABE': 
        t47d = row['T47D'] == 1
    else:
        t47d = 0
    if (not hgc27) and (not mcf7) and (not t47d):
        return 0
    if hgc27 and (not mcf7) and (not t47d):
        return 1
    elif mcf7 and (not hgc27) and (not t47d):
        return 2
    elif hgc27 and mcf7 and (not t47d):
        return 3
    elif t47d and (not hgc27) and (not mcf7):
        return 4
    elif hgc27 and t47d and (not mcf7):
        return 5
    elif mcf7 and t47d and (not hgc27):
        return 6
    else: # all true
        return 7

def get_venn_diagram(df, editor):
    pt = df.pivot_table(index='sgrna', columns='cell_line', values='lt_cutoff')
    pt['venn_set_idx'] = pt.apply(get_venn_set_idx, axis=1, args=(editor,))
    vc = pt['venn_set_idx'].value_counts()
    subsets = []
    if editor == 'ABE':
    # if editor:
        max_range = 8
        labels = ('HGC27', 'MCF7', 'T47D')
        int2id = {0: '000', 1: '100', 2: '010', 3:'110', 4:'001',
          5:'101', 6:'011', 7:'111'}
    else:
        max_range = 4
        labels = ('HGC27', 'MCF7')
        int2id = {0: '00', 1: '10', 2: '01', 3:'11'}

    id2int = {value: key for key, value in int2id.items()}
    
    total = vc.sum()
    percentages = {}
    for i in range(1, max_range):
        try:
            subset_str = int2id[i]
        except:
            import pdb;pdb.set_trace()
        if i in vc:
            percentage = vc[i] / total * 100
            percentages[subset_str] = percentage
            subsets.append(vc[i])
            # text = str(vc[i]) + f'({percentage:.1f}%)' 
            # print(text)
            # subsets.append(text)
        else:
            subsets.append(0)
            percentages[subset_str] = 0

    zero_percentage = vc[0] / total * 100
    zero_text = str(vc[0]) + '\n' + f'({zero_percentage:.1f}%)' 
    one_subsets = [1 for s in subsets]
    if editor == 'ABE':
        plot = venn3(subsets=one_subsets, set_labels=labels)
        # plot = venn3(subsets=subsets, set_labels=labels)
        plt.annotate(zero_text, xy=plot.get_label_by_id('111').get_position() + np.array([0.0, 0.0]), xytext=(-120, -80),
             ha='center', textcoords='offset points')
    else:
        plot = venn2(subsets=one_subsets, set_labels=labels)
        plt.annotate(zero_text, xy=plot.get_label_by_id('11').get_position() + np.array([0.0, 0.0]), xytext=(0, -150),
             ha='center', textcoords='offset points')

    # import pdb;pdb.set_trace()
    for subset_str, percentage in percentages.items():
        # import pdb;pdb.set_trace()
        i = id2int[subset_str]
        try:
            if plot.get_label_by_id(subset_str):  # Check if the subset exists
                # old_text = plot.get_label_by_id(subset).get_text()
                if i in vc:
                    old_text = str(vc[i])
                else:
                    old_text = '0'
                # try:
                #     old_text = str(vc[i])
                # except:
                #     import pdb;pdb.set_trace()
                new_text = old_text + '\n' + f"({percentage:.1f}%)"
                plot.get_label_by_id(subset_str).set_text(new_text)
        except:
            import pdb;pdb.set_trace()

    if editor == 'CBE':
        for text in plot.subset_labels:
            if text:  # Check if the subset label exists
                text.set_fontsize(20)  # Set font size for subset labels

        # Adjust the set label positions
        for i, label in enumerate(plot.set_labels):
            if label:  # Check if the label exists
                # label.set_verticalalignment('bottom')  # Move the label to the top
                label.set_position((i/2-0.25,0.55))
    
    return pt, plot, percentages

In [ ]:
venn_diagram_path = os.path.join(figures_dirpath, 'venn_diagrams')
if not os.path.exists(venn_diagram_path):
    os.mkdir(venn_diagram_path)

percentages_allcl = {'ABE': [], 'CBE': []}
# for gene in no_dup_guide_df['Gene'].unique():
for gene in ['PIK3CA', 'MTOR', 'RPTOR', 'RICTOR', 'PDPK1', 'RHEB', 'MLST8', 'EIF4E', 'RPS6', 'MAPKAP1']:
    print(gene)
    for editor in editors:
        df = no_dup_guide_df[(no_dup_guide_df['Gene'] == gene) & (no_dup_guide_df['editor'] == editor)]
        if editor == 'CBE':
            df = df[df['cell_line'].isin(['HGC27', 'MCF7'])]
        vd_df, plot, percentages = get_venn_diagram(df, editor)
        figure_path = os.path.join(venn_diagram_path, f'{gene}_{editor}.png')
        plt.savefig(figure_path, bbox_inches='tight', dpi=300)
        # plt.show()
        plt.clf()
        if editor == 'ABE':
            allcl = '111'
        else:
            allcl = '11'
        percentages_allcl[editor].append(percentages[allcl])

    # break
        
        # import pdb;pdb.set_trace()

In [ ]:
for editor, l in percentages_allcl.items():
    print(editor, np.min(l), np.max(l))

# Consequence barplots

In [ ]:
conseq_order = ['stop_gained', 'stop_lost', 'start_lost', 
                'splice_acceptor', 'splice_donor', 
                'missense', 
                'splice_polypyrimidine_tract', 'splice_donor_region', 'splice_region',
                'intron',
               'synonymous']

In [ ]:
f = sns.catplot(data=no_dup_guide_df[no_dup_guide_df['is_essential']],
           y='lt_cutoff', 
            hue='main_conseq',
           row='editor',
            row_order=editors,
           col='cell_line',
            col_order=cell_lines,
           kind='bar',
           sharey=False,
           sharex=False,
                errorbar=None,
            hue_order=main_conseq_order)
   

f.set_titles('')
f.set_axis_labels(y_var='Fraction of impactful guides')
# plt.savefig('figures/consq_consv_ess_bar.png', bbox_inches='tight')

In [ ]:
df = no_dup_guide_df[no_dup_guide_df['is_essential']].copy()
df['Hit rate (%)'] = df['lt_cutoff'] * 100

for editor in editors:
    if editor == 'ABE':
        order = ['synonymous', 'missense', 'splice']
    else: # CBE
        order = ['synonymous', 'missense', 'splice', 'stop_gained']
        
    e_df = df[(df['editor'] == editor)
            & (df['main_conseq'].isin(order))].copy()
        
    if editor == 'CBE':
        splice_stop_str = 'splice or\nstop gained'
        e_df['main_conseq'] = e_df['main_conseq'].replace({'splice': splice_stop_str,
                                                           'stop_gained': splice_stop_str})
        order = ['synonymous', 'missense', splice_stop_str]
        
    f = sns.catplot(data=e_df,
           x='Hit rate (%)', 
            y='main_conseq',
                # hue='main_conseq',
        #    row='editor',
        #     row_order=editors,
           col='cell_line',
            col_order=cell_lines,
           kind='bar',
           # sharey=False,
           sharex=False,
                errorbar=None,
            order=order,
               # hue_order=order,
               # palette=[palette[4], palette[5], palette[6]]
                # color = 'royalblue'
                
                color = 'dimgrey',
                
                # color = 'white',
                # linewidth=2.5, 
                # edgecolor="black"
                
                
               )

    for _, editor_axes in zip(editors, f.axes):
        for cell_line, ax in zip(cell_lines, editor_axes):

            cl_e_df = e_df[e_df['cell_line'] == cell_line]

            mean_values = cl_e_df.groupby('main_conseq')['Hit rate (%)'].mean()
            xposlist = [mean_values[e] + mean_values[e] * 0.05 
                        for e in order 
                        if e in mean_values]
            yposlist = range(len(xposlist))
            yposlist = [e for e in yposlist]

            
            counts = cl_e_df.groupby(['main_conseq'])['Hit rate (%)'].count()
            # stringlist = [f'n =\n{counts[e]}' for e in order]
            stringlist = [f'n={counts[e]}' for e in order]
            # stringlist = ['n = 62','n = 19','n = 87','n = 76']
            
            for i in range(len(stringlist)):
                # print(xposlist[i], yposlist[i], stringlist[i])
                ax.text(xposlist[i], yposlist[i], stringlist[i], fontsize='small')

            ax.set_xlim(0, mean_values.max() * 1.35)
            # plt.ylim(-0.05, 1.1)

    f.set_titles('{col_name} - ' + editor)
    f.set_axis_labels(
        # x_var='Hit rate',
                    y_var='Mutation consequence')
    # f.set_xticklabels(rotation=20)
    # plt.legend().remove()
    # plt.show()
    filepath = os.path.join(figures_dirpath, f'consq_consv_ess_bar_{editor}.png')
    plt.savefig(filepath, bbox_inches='tight')

In [ ]:
e_df.groupby(['cell_line', 'editor', 'main_conseq'])['Hit rate (%)'].mean()

In [ ]:
df = no_dup_guide_df[no_dup_guide_df['is_essential']].copy()
df['Hit rate (%)'] = df['lt_cutoff'] * 100
order = ['synonymous', 'missense', 'splice', 'stop_gained']
f = sns.catplot(data=df,
           x='Hit rate (%)', 
            y='main_conseq',
                # hue='main_conseq',
           row='editor',
            row_order=editors,
           col='cell_line',
            col_order=cell_lines,
           kind='bar',
           # sharey=False,
           sharex=False,
                errorbar=None,
            order=order,
               # hue_order=order,
               # palette=[palette[4], palette[5], palette[6]]
                color = 'royalblue'
               )

for editor, editor_axes in zip(editors, f.axes):
    for cell_line, ax in zip(cell_lines, editor_axes):
        
        if editor == 'ABE':
            current_order = ['synonymous', 'missense', 'splice']
        else:
            current_order = order

        cl_e_df = df[(df['cell_line'] == cell_line) 
        & (df['editor'] == editor)
        & (df['main_conseq'].isin(current_order))]

        mean_values = cl_e_df.groupby('main_conseq')['Hit rate (%)'].mean()
        xposlist = [mean_values[e] + mean_values[e] * 0.05 
                    for e in current_order 
                    if e in mean_values]
        yposlist = range(len(xposlist))
        yposlist = [e for e in yposlist]

        
        counts = cl_e_df.groupby(['main_conseq'])['Hit rate (%)'].count()
        # stringlist = [f'n =\n{counts[e]}' for e in order]
        stringlist = [f'n={counts[e]}' for e in current_order]
        # stringlist = ['n = 62','n = 19','n = 87','n = 76']
        
        for i in range(len(stringlist)):
            # print(xposlist[i], yposlist[i], stringlist[i])
            ax.text(xposlist[i], yposlist[i], stringlist[i], fontsize='small')

        ax.set_xlim(0, mean_values.max() * 1.35)
        # plt.ylim(-0.05, 1.1)

f.set_titles('{col_name} - {row_name}')
f.set_axis_labels(
    # x_var='Hit rate',
                 y_var='Mutation consequence')
# f.set_xticklabels(rotation=20)
# plt.legend().remove()
# plt.show()
filepath = os.path.join(figures_dirpath, f'consq_consv_ess_bar_all.png')
plt.savefig(filepath, bbox_inches='tight')

In [ ]:
df[df['main_conseq'].isin(order)].groupby(['editor', 'cell_line', 'main_conseq'])['Hit rate (%)'].mean().round(2)

# General hit rate plot

In [ ]:
df = no_dup_guide_df.copy()
df['Hit rate (%)'] = df['lt_cutoff'] * 100
frac_df = df.groupby(['cell_line', 'editor', 'Gene', 'is_essential'])[['Hit rate (%)', 'gene_effect']].mean().reset_index()
count_df = df.groupby(['cell_line', 'editor', 'Gene', 'is_essential'])[['Hit rate (%)', 'gene_effect']].count().reset_index()
df = frac_df
df = df[df['editor'] == 'ABE']
# df = df[df['editor'] == 'CBE']
df = df.sort_values(['is_essential', 'Hit rate (%)'])
plt.figure(figsize=(16, 8))
f = sns.barplot(data=df,
                y='Hit rate (%)',
               x='Gene',
                hue='cell_line',
               errorbar=None)
# f.set_title(f'{cell_line} - {editor}')
# f.set_ylabel('Hit rate')
plt.xticks(rotation=70)
filepath = os.path.join(figures_dirpath, 'hit_rate_per_target.png')
plt.savefig(filepath, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
ess_target_gene_order = ['RICTOR', 'MAPKAP1', 'MLST8', 'PDPK1', 'RHEB', 'PIK3CA', 'RPS6', 'RPTOR', 'MTOR', 'EIF4E']
palette_name = 'colorblind'
sns.set_palette(palette_name)
palette = sns.color_palette(palette_name)
blue, yellow, green, orange = palette[:4]

In [ ]:
df = no_dup_guide_df.copy()
df = df[df['Consequence'] == 'missense']
df['Hit rate (%)'] = df['lt_cutoff'] * 100
f = sns.catplot(data=df,
                x='Hit rate (%)',
               y='Gene',
                # hue='cell_line',
                row='editor',
                row_order=editors,
                col='cell_line',
                col_order=cell_lines,
                kind='bar',
                order=ess_target_gene_order,
               errorbar=None,
               sharex=False,
               color=blue)

col_name = 'Gene'
order = ess_target_gene_order
for editor, editor_axes in zip(editors, f.axes):
    for cell_line, ax in zip(cell_lines, editor_axes):

        cl_e_df = df[(df['cell_line'] == cell_line) 
        & (df['editor'] == editor)
        & (df[col_name].isin(order))]

        mean_values = cl_e_df.groupby(col_name)['Hit rate (%)'].mean()
        xposlist = [mean_values[e] + mean_values[e] * 0.05 for e in order]
        yposlist = range(len(xposlist))
        yposlist = [e + 0.10 for e in yposlist]

        
        counts = cl_e_df.groupby([col_name])['Hit rate (%)'].count()
        stringlist = [f'n={counts[e]}' for e in order]
        
        for i in range(len(stringlist)):
            # print(xposlist[i], yposlist[i], stringlist[i])
            ax.text(xposlist[i], yposlist[i], stringlist[i], fontsize='small')

        ax.set_xlim(0, mean_values.max() * 1.5)

f.set_titles('{col_name} - {row_name}')
filepath = os.path.join(figures_dirpath, 'hit_rate_per_target_missense_only.png')
plt.savefig(filepath, bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
abe_frac_df = frac_df[frac_df['editor'] == 'ABE']
abe_count_df = count_df[count_df['editor'] == 'ABE']

In [ ]:
vc = abe_frac_df[['is_essential', 'Gene', 'cell_line']].drop_duplicates().groupby('Gene')['is_essential'].sum()
ess_genes = vc[vc == 3].index

In [ ]:
pt_frac = abe_frac_df.pivot_table(index=['Gene'], columns='cell_line', values='Hit rate (%)')
pt_frac = pt_frac.reset_index()
pt_frac['max'] = pt_frac[cell_lines].max(axis=1)
pt_frac['is_essential'] = pt_frac['Gene'].apply(lambda g: g in ess_genes)

In [ ]:
pt_count = abe_count_df.pivot_table(index=['Gene'], columns='cell_line', values='Hit rate (%)')
pt_count = pt_count.reset_index()
pt_count['count'] = pt_count[cell_lines].max(axis=1)
# pt['is_essential'] = pt['Gene'].apply(lambda g: g in ess_genes)

In [ ]:
pt_frac.merge(pt_count[['Gene', 'count']], on='Gene').sort_values(['is_essential', 'max'], ascending=False).set_index('Gene')# [cell_lines]

# Hit rate vs gene effect

In [ ]:
mean_grouped_mdf = no_dup_guide_df.groupby(['cell_line', 'editor', 'Gene'])[['lt_cutoff', 'gene_effect']].mean().reset_index()
count_grouped_mdf = no_dup_guide_df.groupby(['cell_line', 'editor', 'Gene'])[['lt_cutoff', 'gene_effect']].count().reset_index()
count_grouped_mdf = count_grouped_mdf.rename({'lt_cutoff': 'count'}, axis=1).drop('gene_effect', axis=1)
grouped_mdf = mean_grouped_mdf.merge(count_grouped_mdf, on=['cell_line', 'editor', 'Gene'])

In [ ]:
grouped_mdf['Hit rate (%)'] = grouped_mdf['lt_cutoff'] * 100

In [ ]:
for cell_line in cell_lines:
    for editor in editors:
        subset_df = grouped_mdf.query(f'(cell_line == "{cell_line}") and (editor == "{editor}")')
        subset_df[['Gene', 'gene_effect', 'Hit rate (%)']].to_csv(f'hit_rate_vs_gene_effect_{cell_line}_{editor}.csv', 
                                                          index=False)
        print(cell_line, editor)
        print(pearsonr(subset_df['gene_effect'], subset_df['Hit rate (%)']))

In [ ]:
ndg_filtered_df = no_dup_guide_df[no_dup_guide_df['main_conseq'].isin(main_conseq_order)]
mean_grouped_mdf = ndg_filtered_df.groupby(['cell_line', 'editor', 'Gene'])[['lt_cutoff', 'gene_effect']].mean().reset_index()
count_grouped_mdf = ndg_filtered_df.groupby(['cell_line', 'editor', 'Gene'])[['lt_cutoff', 'gene_effect']].count().reset_index()
count_grouped_mdf = count_grouped_mdf.rename({'lt_cutoff': 'count'}, axis=1).drop('gene_effect', axis=1)
grouped_mdf = mean_grouped_mdf.merge(count_grouped_mdf, on=['cell_line', 'editor', 'Gene'])

In [ ]:
grouped_mdf['Hit rate (%)'] = grouped_mdf['lt_cutoff'] * 100

In [ ]:
for cell_line in cell_lines:
    for editor in editors:
        subset_df = grouped_mdf.query(f'(cell_line == "{cell_line}") and (editor == "{editor}")')
        subset_df[['Gene', 'gene_effect', 'Hit rate (%)']].to_csv(f'hit_rate_vs_gene_effect_{cell_line}_{editor}_filtered.csv', 
                                                          index=False)

In [ ]:
f = sns.relplot(data=grouped_mdf,
           x='lt_cutoff',
           y='gene_effect',
           row='editor',
           row_order=editors,
           col='cell_line',
           col_order=cell_lines,
            size='count',
           facet_kws=dict(sharex=False,
           sharey=False))
f.set_axis_labels(x_var='Hit rate', y_var='DEPMAP gene effect')
f.set_titles('{col_name} - {row_name}')
filepath = os.path.join(figures_dirpath, 'fraction_hits_vs_depmap_gene_effect.png')
plt.savefig(filepath, bbox_inches='tight', dpi=300)

In [ ]:
sns.relplot(data=grouped_mdf,
           x='count',
           y='lt_cutoff',
           row='editor',
           row_order=editors,
           col='cell_line',
           col_order=cell_lines,
           facet_kws=dict(sharex=False,
           sharey=False))

# Variant class plots

In [ ]:
missense_df['Variant class'] = missense_df['Variant class'].astype(str)

In [ ]:
missense_df['Variant class'] = missense_df['Variant class'].replace('nan', 'unknown')

In [ ]:
missense_df.groupby(['editor', 'Variant class'])['sgrna'].count().reset_index().pivot(index='Variant class', columns=['editor'], values='sgrna')

In [ ]:
missense_df['Variant class'].value_counts(dropna=False).reset_index()

In [ ]:
def get_pie_chart(df):
    cons_vc = df['Variant class'].value_counts()
    cons_vc = cons_vc[cons_vc > 0]
    sizes = []
    labels = []
    for general_cons in list(reversed(variant_order)):
        label = general_cons
        size = 0
        for actual_cons in cons_vc.index:
            if general_cons in actual_cons:
                size += cons_vc[actual_cons]
        if size > 0:
            labels.append(label)
            sizes.append(size)
    fig, ax = plt.subplots()
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', pctdistance=.85)

In [ ]:
for editor in editors:
    get_pie_chart(missense_df[(missense_df['editor'] == editor) 
                  & (missense_df['is_essential'])
                  ])
    plt.title(editor)
    plt.savefig(f'figures/pie_chart_variant_class_{editor}.png', bbox_inches='tight')

In [ ]:
order

In [ ]:
df = missense_df[missense_df['is_essential']].copy()
df['Hit rate (%)'] = df['lt_cutoff'] * 100
f = sns.catplot(data=df,
           x='Hit rate (%)', 
            y='Variant class',
            # hue='Variant class',
           row='editor',
            row_order=editors,
            col_order=cell_lines,
           col='cell_line',
           kind='bar',
           sharex=False,
            # sharey=False,
            
            errorbar=None,
           #  order=['unknown'] + list(reversed(variant_order)),
           # hue_order= ['unknown'] + list(reversed(variant_order))
             order=list(reversed(variant_order)),
           # hue_order=list(reversed(variant_order)),
                # color=palette[2]
                color='steelblue'
           )

order = list(reversed(variant_order))
for editor, editor_axes in zip(editors, f.axes):
    for cell_line, ax in zip(cell_lines, editor_axes):

        cl_e_df = df[(df['cell_line'] == cell_line) 
        & (df['editor'] == editor)
        & (df['Variant class'].isin(order))]

        mean_values = cl_e_df.groupby('Variant class')['Hit rate (%)'].mean()
        xposlist = [mean_values[e] + mean_values[e] * 0.05 for e in order]
        yposlist = range(len(xposlist))
        yposlist = [e for e in yposlist]

        
        counts = cl_e_df.groupby(['Variant class'])['Hit rate (%)'].count()
        # stringlist = [f'n =\n{counts[e]}' for e in order]
        stringlist = [f'n={counts[e]}' for e in order]
        # stringlist = ['n = 62','n = 19','n = 87','n = 76']
        
        for i in range(len(stringlist)):
            # print(xposlist[i], yposlist[i], stringlist[i])
            ax.text(xposlist[i], yposlist[i], stringlist[i], fontsize='small')

        ax.set_xlim(0, mean_values.max() * 1.35)
        # plt.ylim(-0.05, 1.1)


f.set_titles('{col_name} - {row_name}')
# f.set_axis_labels(y_var='Hit rate')
f.set_axis_labels(
    # x_var='Hit rate',
                 y_var='Missense predicted impact')
filepath = os.path.join(figures_dirpath, 'variant_class_bar_missense_only.png')
plt.savefig(filepath, bbox_inches='tight', dpi=300)

In [ ]:
df[df['Variant class'].isin(order)].groupby(['editor', 'cell_line', 'Variant class'])['Hit rate (%)'].mean().round(2)

# Mutation analysis

In [ ]:
subset_df = missense_df[(missense_df['cell_line'] == 'HGC27') & (missense_df['is_essential'])]

In [ ]:
mut_count = subset_df.groupby(['editor', 'AA_Mutation'])['lt_cutoff'].count()
mut_count = mut_count.reset_index()
mut_count = mut_count.rename({'lt_cutoff': 'count'}, axis=1)

total_mut = subset_df.groupby(['editor'])['lt_cutoff'].count()
total_mut = total_mut.reset_index()
total_mut = total_mut.rename({'lt_cutoff': 'total'}, axis=1)

mut_count = mut_count.merge(total_mut, on='editor')

mut_count['fraction'] = mut_count['count'] / mut_count['total']

lt_cutoff = subset_df.groupby(['editor', 'AA_Mutation'])['lt_cutoff'].mean()
lt_cutoff = lt_cutoff.reset_index()

lt_cutoff = lt_cutoff.merge(mut_count, on=['editor', 'AA_Mutation'])

In [ ]:
for editor in editors:
    editor_df = lt_cutoff[lt_cutoff['editor'] == editor].copy()
    editor_df['Hit rate'] = editor_df['lt_cutoff']
    aa_mutation_name = 'AA_Mutation'

    fig, ax = plt.subplots(1, 2, figsize=(10, 8))
    
    # plt.figure(figsize=(5, 8))
    sns.barplot(data=editor_df.sort_values('lt_cutoff', ascending=False),
               x='Hit rate',
               y=aa_mutation_name,
                errorbar=None,
               ax=ax[0])
    # plt.show()

    # plt.figure(figsize=(5, 8))
    sns.barplot(data=editor_df.sort_values('fraction', ascending=False),
               x='fraction',
               y=aa_mutation_name,
                errorbar=None,
               ax=ax[1])
    ax[1].set_ylabel(None)
    filepath = os.path.join(figures_dirpath, f'barplot_aa_mutation_lt_cutoff_fraction_{editor}.png')
    plt.savefig(filepath, bbox_inches='tight', dpi=300)
    plt.show()
    
    sns.scatterplot(data=editor_df,
                   x='fraction',
                   y='Hit rate')
    plt.show()

# Tests "waterfall plots"

In [ ]:
domains = {'PI3K-ABD': [16, 105],
          'PI3K-RBD': [187, 289],
          'C2 PI3K-type': [330, 487],
          'PIK helical': [517, 694],
          'PI3K-PI4K catalytic': [765, 1051],
          'G-loop': [771, 777],
          'Catalytic loop': [912, 920],
          'Activation loop': [931, 957]}
colors = ['blue', 'lightblue', 'purple', 'pink', 'green', 'red', 'orange', 'yellow']
lw = 1

editor = 'ABE'
gene = 'PIK3CA'
df = no_dup_position_df[
# (no_dup_position_df['cell_line'] == 'cell_line')
# & 
(no_dup_position_df['editor'] == editor)
&
(no_dup_position_df['Gene'] == gene)
&
(no_dup_position_df['Consequence'] == 'missense')]

fig, ax = plt.subplots(figsize=(12, 5))

sns.scatterplot(data=df,
               x='Protein_position',
               y='LFC',
               hue='cell_line',
                style='cell_line',
                size=5.0,
               ax=ax)
for (domain_name, domain_limits), color in zip(domains.items(), colors):
    plt.axvline(x=domain_limits[0], color=color, lw=lw)
    plt.axvline(x=domain_limits[1], color=color, lw=lw)
plt.xlim(-10, 1080)
plt.show()

In [ ]:
domains = {'N-HEAT': [1, 900],
          'C-HEAT': [900, 1345],
          'TPR 1': [1346, 1382],
          'FAT': [1382, 1982],
          'FKBP1A/rapamycin interaction': [2012, 2144],
          'PI3K/PI4K catalytic': [2156, 2469],
          'FATC': [2517, 2549]}
colors = ['blue', 'lightblue', 'purple', 'pink', 'green', 'red', 'orange', 'yellow']
lw = 1

editor = 'ABE'
gene = 'MTOR'
df = no_dup_position_df[
# (no_dup_position_df['cell_line'] == 'cell_line')
# & 
(no_dup_position_df['editor'] == editor)
&
(no_dup_position_df['Gene'] == gene)
&
(no_dup_position_df['Consequence'] == 'missense')]

fig, ax = plt.subplots(figsize=(12, 5))

sns.scatterplot(data=df,
               x='Protein_position',
               y='LFC',
               hue='cell_line',
                style='cell_line',
                size=5.0,
               ax=ax)
for (domain_name, domain_limits), color in zip(domains.items(), colors):
    plt.axvline(x=domain_limits[0], color=color, lw=lw)
    plt.axvline(x=domain_limits[1], color=color, lw=lw)
# plt.xlim(-10, 1080)
plt.show()

# Correlation between cell lines

In [ ]:
test = no_dup_guide_df.merge(no_dup_guide_df, on=['sgrna', 'editor'])

In [ ]:
for editor in editors:
    for i, cell_line_x in enumerate(cell_lines):
        for cell_line_y in cell_lines[i+1:]:
            print(editor, cell_line_x, cell_line_y)
            t2 = test[(test['cell_line_x'] == cell_line_x) 
            & (test['cell_line_y'] == cell_line_y)
            & (test['editor'] == editor)]

            pearson_coeff = pearsonr(t2['LFC_x'].values, t2['LFC_y'].values).statistic
            spearman_coeff = spearmanr(t2['LFC_x'].values, t2['LFC_y'].values).statistic

            ax = sns.scatterplot(data=t2,
                                    x='LFC_x',
                                   y='LFC_y')
            print('Pearson = ', pearson_coeff, '; Spearman = ', spearman_coeff)
            plt.show()

# Other data exploration

In [ ]:
subset_df = with_mut_df[with_mut_df['Consequence'] == 'missense'].copy()
subset_df['AA_Mutation'] = subset_df['Original_AA'] + '/' + subset_df['Modified_AA']
unique_mut = subset_df.groupby(['sgrna', 'editor', 'Protein_position'])['AA_Mutation'].unique()

In [ ]:
unique_mut = with_mut_df.groupby(['sgRNA_ID', 'editor', 'Protein_position'])['Codons'].unique()

In [ ]:
subset_df = with_mut_df.copy()
subset_df = subset_df[subset_df['Consequence'] == 'missense']
subset_df = subset_df.drop_duplicates(['cell_line', 'editor', 'sgrna', 'Protein_position', 'Original_AA', 'Modified_AA'])
subset_df['AA_Mutation'] = subset_df['Original_AA'] + '/' + subset_df['Modified_AA']

In [ ]:
old_vc = subset_df[subset_df['is_essential']]['Original_AA'].value_counts(dropna=False)
new_vc = subset_df[subset_df['is_essential']]['Modified_AA'].value_counts(dropna=False)

In [ ]:
aa_change_order = sorted(subset_df['AA_Mutation'].unique())

In [ ]:
subset_df = subset_df.sort_values(['AA_Mutation'])

In [ ]:
with_mut_df[(with_mut_df['Gene'] == 'MTOR') 
& (with_mut_df['Protein_position'] == 2108)].groupby('sgrna')['Amino_acids'].unique()

In [ ]:
with_mut_df[(with_mut_df['sgrna'] == '1347PIK3CA') 
& (with_mut_df['editor'] == 'ABE')
][['cell_line', 'Consequence', 'Direction', 'Strand', 'Amino_acids', 'editor', 'ew_position', 'LFC']]

In [ ]:
subset_df['AA_Mutation'].value_counts()

# Using annotations

In [ ]:
annotations_dfs=[]
annotations_dirpath = cfg['uniprot_annotations_dirpath']
for gene in target_genes:
    annotations_df_filepath = os.path.join(annotations_dirpath, f'{gene}_annotations.csv')
    annotations_df = pd.read_csv(annotations_df_filepath)
    annotations_df['uniprot_rno'] = annotations_df['uniprot_rno'].apply(lambda x: int(x))
    annotations_df['Gene'] = gene
    
    # annotated=pd.merge(cg, deplex[deplex['Gene']==gene], on=['uniprot_rno'], how='inner')
    # print(annotated['annotation_type'].value_counts())
    
    annotations_dfs.append(annotations_df)

In [ ]:
annotations_df = pd.concat(annotations_dfs)
# annotations_df['uniprot_rno'] = annotations_df['uniprot_rno'].astype(str)
# annotations_df['Gene_RNO'] = annotations_df['Gene'] + '_' + annotations_df['uniprot_rno'].astype(str)

In [ ]:
annotations_df.head()

In [ ]:
annotations_df = annotations_df.rename({'uniprot_rno': 'Protein_position'}, axis=1)

In [ ]:
with_annot_df = missense_df.merge(annotations_df, 
                                    on=['Gene', 'Protein_position'],
                                   how='left')
with_annot_df = with_annot_df.sort_values(['cell_line', 'editor', 'Gene', 'Protein_position', 'annotation_type', 'Consequence', 'LFC'])
with_annot_df = with_annot_df.drop_duplicates(['cell_line', 'editor', 'Gene', 'Protein_position', 'annotation_type'])

In [ ]:
with_annot_df.shape

In [ ]:
def get_clean_annotation(row):
    annotation_type = row['annotation_type']
    annotation_note = row['annotation_note']
    if annotation_type in ['Domain', 'Region', 'Compositional bias']:
        # if (annotation_note != 'Disordered') and not ('loop' in annotation_note):
        if not ('loop' in annotation_note):
            return annotation_note
        else:
            return np.nan
    elif annotation_type in ['Repeat']:
        return annotation_type
    else:
        return np.nan

In [ ]:
annotations_df['annotation_type'].value_counts()

In [ ]:
clean_annotation_df = annotations_df.copy()
clean_annotation_df['clean_annotation'] = clean_annotation_df.apply(get_clean_annotation, axis=1)

In [ ]:
clean_annotation_df = clean_annotation_df.sort_values(['clean_annotation', 'Protein_position'])

# Secondary structure

In [ ]:
SS_KEYWORDS = ['Helix', 'Beta strand', 'Turn']
ss_order = SS_KEYWORDS + ['None']
def get_ss(row):
    annotation_type = row['annotation_type']
    if annotation_type in SS_KEYWORDS:
        return annotation_type
    else:
        return 'None'

In [ ]:
ss_df = annotations_df.copy()
ss_df['secondary_structure'] = ss_df.apply(get_ss, axis=1)
ss_df['secondary_structure'] = pd.Categorical(ss_df['secondary_structure'], ss_order)
ss_df = ss_df.sort_values('secondary_structure')
ss_df = ss_df.drop_duplicates(['Gene', 'Protein_position']) # should remove Nones

In [ ]:
missense_df_ss = missense_df.merge(ss_df[['Protein_position', 'Gene', 'secondary_structure']],
                                                 on=['Protein_position', 'Gene'],
                                                how='left')
# missense_df_ss['secondary_structure'] = missense_df_ss['secondary_structure'].fillna('Not in CDS')

In [ ]:
missense_df_ss['secondary_structure'].value_counts()

In [ ]:
df = missense_df_ss[missense_df_ss['is_essential']].copy()
df['Hit rate (%)'] = df['lt_cutoff'] * 100
# ss_order = SS_KEYWORDS + ['None', 'Not in CDS']
f = sns.catplot(data=df,
           x='Hit rate (%)', 
            y='secondary_structure',
           row='editor',
            row_order=editors,
            col_order=cell_lines,
           col='cell_line',
           kind='bar',
           sharex=False,
            errorbar=None,
             order=ss_order,
           )

order = ss_order
for editor, editor_axes in zip(editors, f.axes):
    for cell_line, ax in zip(cell_lines, editor_axes):

        cl_e_df = df[(df['cell_line'] == cell_line) 
        & (df['editor'] == editor)
        & (df['secondary_structure'].isin(order))]

        mean_values = cl_e_df.groupby('secondary_structure')['Hit rate (%)'].mean()
        xposlist = [mean_values[e] + mean_values[e] * 0.05 for e in order]
        yposlist = range(len(xposlist))
        yposlist = [e for e in yposlist]

        
        counts = cl_e_df.groupby(['secondary_structure'])['Hit rate (%)'].count()
        # stringlist = [f'n =\n{counts[e]}' for e in order]
        stringlist = [f'n={counts[e]}' for e in order]
        # stringlist = ['n = 62','n = 19','n = 87','n = 76']
        
        for i in range(len(stringlist)):
        #     # print(xposlist[i], yposlist[i], stringlist[i])
            ax.text(xposlist[i], yposlist[i], stringlist[i], fontsize='small')

        ax.set_xlim(0, mean_values.max() * 1.35)


f.set_titles('{col_name} - {row_name}')
# f.set_axis_labels(y_var='Hit rate')
f.set_axis_labels(
    # x_var='Hit rate',
                 y_var='Secondary structure')
filepath = os.path.join(figures_dirpath, 'ss_bar_missense_only.png')
plt.savefig(filepath)

In [ ]:
colors = sns.color_palette('colorblind') + ['grey']

# Waterfall plots

In [ ]:
waterfall_dirpath = os.path.join(figures_dirpath, 'waterfall')
if not os.path.exists(waterfall_dirpath):
    os.makedirs(waterfall_dirpath)

editor = 'ABE'
for gene in sorted(target_genes):
    for cell_line in cell_lines:
        df = no_dup_position_df[
        (no_dup_position_df['cell_line'] == cell_line)
        & 
        (no_dup_position_df['editor'] == editor)
        &
        (no_dup_position_df['Gene'] == gene)
        &
        (no_dup_position_df['Consequence'] == 'missense')]
        
        fig, ax = plt.subplots(figsize=(12, 5))
    
        if len(df) == 0:
            continue
        
        sns.scatterplot(data=df,
                       x='Protein_position',
                       y='LFC',
                       hue='lt_cutoff',
                        style='cell_line',
                        size=5.0,
                       ax=ax)
        cutoff = df['lfc_cutoff'].values[0]
        ax.axhline(cutoff, color='k')
        # sns.move_legend(ax, loc='upper left', bbox_to_anchor=(1,1))
    
        ps = []
        annots = []
        gene_annotation_df = clean_annotation_df[clean_annotation_df['Gene'] == gene]
        gene_annotation_df = gene_annotation_df.sort_values('Protein_position')
        for i, (annotation_note, grouped_df) in enumerate(gene_annotation_df.groupby('clean_annotation', sort=False)):
            if not isinstance(annotation_note, str) and np.isnan(annotation_note):
                continue
            color = colors[i]
            x = grouped_df['Protein_position'].values
            sorted_x = sorted(x)
            current_values = [sorted_x[0]]

            low_y = 1 + i * 0.2
            high_y = 1 + (i + 1) * 0.2
            
            for v in sorted_x[1:]:
                previous_value = current_values[-1]
                if v != previous_value + 1:
                    current_values = np.array(current_values)
                    where = np.ones_like(current_values)
                    ax.fill_between(current_values, low_y, high_y, where=where,
                                color=color, alpha=0.2, 
                                    # transform=ax.get_xaxis_transform()
                                   )
                    # p1 = ax.plot(x, y, color='b', linewidth=3)
                    p2 = ax.fill(np.nan, np.nan, color=color, alpha=0.2)
                    current_values = [v]
                else:
                    current_values.append(v)
            current_values = np.array(current_values)
            where = np.ones_like(current_values)
            ax.fill_between(current_values, low_y, high_y, where=where,
                        color=color, alpha=0.2, 
                            # transform=ax.get_xaxis_transform()
                           )
            # p1 = ax.plot(x, y, color='b', linewidth=3)
            p2 = ax.fill(np.nan, np.nan, color=color, alpha=0.2)
            ps.append((p2[0]))
            annots.append(annotation_note)
            # if gene == 'AKT1S1':
            #     import pdb;pdb.set_trace()
    
        ax.legend(reversed(ps), reversed(annots), loc='upper left', bbox_to_anchor=(1,1))
        # where = [True, True]
        # for (domain_name, domain_limits), color in zip(domains.items(), colors):
        #     # plt.axvline(x=domain_limits[0], color=color, lw=lw)
        #     # plt.axvline(x=domain_limits[1], color=color, lw=lw)
        #     x = domain_limits
        #     ax.fill_between(x, 0, 1, where=where,
        #                 color=color, alpha=0.2, transform=ax.get_xaxis_transform())
        # plt.xlim(-10, 1080)
        plt.title(f'{gene} - {cell_line}')
        filepath = os.path.join(waterfall_dirpath, f'{gene}_{cell_line}_missense_waterfall.png')
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()
        # plt.show()

In [ ]:
no_dup_position_df_abe = no_dup_position_df_filtered.query("editor == 'ABE'")
n_hits = no_dup_position_df_abe.groupby(['Gene', 'Protein_position'])['lt_cutoff'].sum()
hit_in_all_cl = n_hits >= 3
hit_in_all_cl.name = 'hit_in_all_cell_line'
no_dup_position_df_abe = no_dup_position_df_abe.merge(hit_in_all_cl.reset_index(),
                                                on=['Gene', 'Protein_position'])

no_dup_position_df_cbe = no_dup_position_df_filtered.query("editor == 'CBE' and cell_line in ['MCF7', 'HGC27']")
n_hits = no_dup_position_df_cbe.groupby(['Gene', 'Protein_position'])['lt_cutoff'].sum()
hit_in_all_cl = n_hits >= 2
hit_in_all_cl.name = 'hit_in_all_cell_line'
no_dup_position_df_cbe = no_dup_position_df_cbe.merge(hit_in_all_cl.reset_index(),
                                                on=['Gene', 'Protein_position'])

no_dup_position_df_nh = pd.concat([no_dup_position_df_abe, no_dup_position_df_cbe])

In [ ]:
for gene in sorted(target_genes):
            
    df = no_dup_position_df_nh[
    (no_dup_position_df_nh['Gene'] == gene)
    &
    (no_dup_position_df_nh['Consequence'] == 'missense')]
    
    fig, ax = plt.subplots(
        figsize=(15, 5)
    )

    if len(df) == 0:
        continue
    
    scatter = sns.scatterplot(data=df,
                              x='Protein_position',
                              y='zscore',
                              hue='hit_in_all_cell_line',
                              style='cell_line',
                              size=5.0,
                              ax=ax)
    cutoff = df['lfc_cutoff'].values[0]
    # ax.axhline(cutoff, color='k')
    # sns.move_legend(ax, loc='upper left', bbox_to_anchor=(1,1))

    ps = []
    annots = []
    gene_annotation_df = clean_annotation_df[clean_annotation_df['Gene'] == gene]
    gene_annotation_df = gene_annotation_df.sort_values('Protein_position')

    groupby = gene_annotation_df.groupby('clean_annotation', sort=False)
    n_annot = len(groupby)

    for i, (annotation_note, grouped_df) in enumerate(groupby):
        if not isinstance(annotation_note, str) and np.isnan(annotation_note):
            continue
        color = colors[i]
        x = grouped_df['Protein_position'].values
        sorted_x = sorted(x)
        current_values = [sorted_x[0]]

        low_y = 2 + (n_annot - i) * 0.2
        high_y = 2 + (n_annot - i + 1) * 0.2
        
        for v in sorted_x[1:]:
            previous_value = current_values[-1]
            if v != previous_value + 1:
                current_values = np.array(current_values)
                where = np.ones_like(current_values)
                ax.fill_between(current_values, low_y, high_y, where=where,
                                color=color, alpha=0.2)
                p2 = ax.fill(np.nan, np.nan, color=color, alpha=0.2)
                current_values = [v]
            else:
                current_values.append(v)
        current_values = np.array(current_values)
        where = np.ones_like(current_values)
        ax.fill_between(current_values, low_y, high_y, where=where,
                        color=color, alpha=0.2)
        p2 = ax.fill(np.nan, np.nan, color=color, alpha=0.2)
        ps.append((p2[0]))

        if 'rapamycin' in annotation_note:
            annotation_note = 'Interaction with FKBP1A/rapamycin'
        
        annots.append(annotation_note)

    # First legend for the annotations
    legend1 = ax.legend(ps, annots, title='Annotation', loc='upper left', 
                        bbox_to_anchor=(1.05, 1.10)
                       )
    ax.add_artist(legend1)

    # Create a separate legend for the cell line shapes
    handles, labels = scatter.get_legend_handles_labels()
    cell_line_handles = handles[-len(df['cell_line'].unique()):]
    cell_line_labels = labels[-len(df['cell_line'].unique()):]
    legend2 = ax.legend(cell_line_handles, cell_line_labels, title='Cell lines', loc='lower left', 
                        bbox_to_anchor=(1.05, -0.25),
                        ncol=2
                       )

    plt.title(f'{gene}')
    plt.xlabel('Position')
    plt.ylabel('Z-score')
    plt.tight_layout()
    filepath = os.path.join(waterfall_dirpath, f'{gene}_grouped.png')
    plt.savefig(filepath, dpi=300, 
                bbox_inches='tight',
                bbox_extra_artists=[legend1, legend2]
               )
    plt.close()
    # plt.show()
    # break

In [ ]:
for gene in sorted(target_genes):
    for editor in editors:
        # for cell_line in cell_lines:
        if editor == 'ABE':
            current_cell_lines = cell_lines
        else:
            current_cell_lines = ['HGC27', 'MCF7']
            
        df = no_dup_position_df_nh[
        (no_dup_position_df_nh['cell_line'].isin(current_cell_lines))
        & 
        (no_dup_position_df_nh['editor'] == editor)
        &
        (no_dup_position_df_nh['Gene'] == gene)
        &
        (no_dup_position_df_nh['Consequence'] == 'missense')]
        
        fig, ax = plt.subplots(figsize=(12, 5))
    
        if len(df) == 0:
            continue
        
        sns.scatterplot(data=df,
                       x='Protein_position',
                       y='zscore',
                       # hue='annotation_note',
                        hue='hit_in_all_cell_line',
                        style='cell_line',
                        size=5.0,
                       ax=ax)
        cutoff = df['lfc_cutoff'].values[0]
        # ax.axhline(cutoff, color='k')
        # sns.move_legend(ax, loc='upper left', bbox_to_anchor=(1,1))
    
        ps = []
        annots = []
        gene_annotation_df = clean_annotation_df[clean_annotation_df['Gene'] == gene]
        gene_annotation_df = gene_annotation_df.sort_values('Protein_position')

        groupby = gene_annotation_df.groupby('clean_annotation', sort=False)
        n_annot = len(groupby)
    
        for i, (annotation_note, grouped_df) in enumerate(groupby):
            if not isinstance(annotation_note, str) and np.isnan(annotation_note):
                continue
            color = colors[i]
            x = grouped_df['Protein_position'].values
            sorted_x = sorted(x)
            current_values = [sorted_x[0]]

            low_y = 2 + (n_annot - i) * 0.2
            high_y = 2 + (n_annot - i + 1) * 0.2
            
            for v in sorted_x[1:]:
                previous_value = current_values[-1]
                if v != previous_value + 1:
                    current_values = np.array(current_values)
                    where = np.ones_like(current_values)
                    ax.fill_between(current_values, low_y, high_y, where=where,
                                color=color, alpha=0.2, 
                                    # transform=ax.get_xaxis_transform()
                                   )
                    # p1 = ax.plot(x, y, color='b', linewidth=3)
                    p2 = ax.fill(np.nan, np.nan, color=color, alpha=0.2)
                    current_values = [v]
                else:
                    current_values.append(v)
            current_values = np.array(current_values)
            where = np.ones_like(current_values)
            ax.fill_between(current_values, low_y, high_y, where=where,
                        color=color, alpha=0.2, 
                            # transform=ax.get_xaxis_transform()
                           )
            # p1 = ax.plot(x, y, color='b', linewidth=3)
            p2 = ax.fill(np.nan, np.nan, color=color, alpha=0.2)
            ps.append((p2[0]))

            if 'rapamycin' in annotation_note:
                annotation_note = 'Interaction with FKBP1A/rapamycin'
            
            annots.append(annotation_note)
            # if gene == 'AKT1S1':
            #     import pdb;pdb.set_trace()
    
        ax.legend(ps, annots, loc='upper left', bbox_to_anchor=(1,1))
        # where = [True, True]
        # for (domain_name, domain_limits), color in zip(domains.items(), colors):
        #     # plt.axvline(x=domain_limits[0], color=color, lw=lw)
        #     # plt.axvline(x=domain_limits[1], color=color, lw=lw)
        #     x = domain_limits
        #     ax.fill_between(x, 0, 1, where=where,
        #                 color=color, alpha=0.2, transform=ax.get_xaxis_transform())
        # plt.xlim(-10, 1080)
        plt.title(f'{gene} - {editor}')
        filepath = os.path.join(waterfall_dirpath, f'{gene}_{editor}_grouped.png')
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        plt.close()

# With annotation summary

In [ ]:
def get_annotation_summary(annotation_type: str):
    if annotation_type in ['Helix', 'Beta strand', 'Turn']:
        return 'secondary_structure'
    # elif annotation_type in ['Site', 'Active site', 'Binding site']:
    elif annotation_type in ['Binding site']:
        return 'Binding site'
    else:
        return annotation_type

with_annot_df['annotation_summary'] = with_annot_df['annotation_type'].apply(get_annotation_summary)

In [ ]:
annotation_order = ['Chain', 'secondary_structure', 'Disulfide bond', 'Binding site']

In [ ]:
with_annot_df['annotation_summary'].value_counts()

In [ ]:
subset_df = with_annot_df.drop_duplicates(['Protein_position', 'cell_line', 'editor', 'annotation_summary'])

In [ ]:
with_annot_df[with_annot_df['annotation_summary'] == 'Disulfide bond'].sort_values('LFC')

In [ ]:
# subset_df = with_annot_df.drop_duplicates(['Protein_position', 'cell_line', 'editor', 'annotation_type'])
sns.catplot(data=subset_df[subset_df['is_essential']], #[with_inter_df['consq_consv'] == 'missense_non_conservative'],
           y='lt_cutoff', 
            hue='annotation_summary',
           row='editor',
            row_order=editors,
           col='cell_line',
            col_order=cell_lines,
           kind='bar',
            errorbar=None,
           sharey= False,
           # sharex= False,
            hue_order=['Chain', 'Binding site'],
           # common_norm=False
           )
filepath = os.path.join(figures_dirpath, 'annotations_bar.png')
plt.savefig(filepath, bbox_inches='tight', dpi=300)

In [ ]:
subset_df = with_annot_df.drop_duplicates(['Protein_position', 'cell_line', 'editor', 'annotation_type'])
sns.displot(data=subset_df[subset_df['is_essential']], #[with_inter_df['consq_consv'] == 'missense_non_conservative'],
           x='LFC', 
            hue='annotation_summary',
           col='editor',
           row='cell_line',
           kind='kde',
           facet_kws={'sharey': False,
                     'sharex': False},
            hue_order=['Chain', 'secondary_structure', 'Binding site', 'Disulfide bond', 'Cross-link'],
           common_norm=False)

In [ ]:
sns.displot(data=with_annot_df[(with_annot_df['is_essential']) & (with_annot_df['is_conservative'] == False)], 
           x='LFC', 
            hue='annotation_type',
           col='editor',
           row='cell_line',
           kind='kde',
           facet_kws={'sharey': False,
                     'sharex': False},
            hue_order=annotation_order,
           common_norm=False)
filepath = os.path.join(figures_dirpath, 'annotations_kde_non_conservative.png')
plt.savefig(filepath, bbox_inches='tight', dpi=300)